In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import dask
from dask import dataframe as dd
from geopy import distance

import glob
from tqdm import tqdm

In [2]:
import dask_jobqueue
import distributed

# this first part is checking you're in the right environment
if "client" in locals():
    client.close()
    del client
if "cluster" in locals():
    cluster.close()

# this is where we set up the cluster, your own compute system if you will 
cluster = dask_jobqueue.PBSCluster(
    cores=1,  # The number of cores you want
    memory="40GB",  # Amount of memory
    processes=1,  # How many processes
    queue="casper",  # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    # log_directory="/glade/scratch/dcherian/dask/",  # Use your local directory
    resource_spec="select=1:ncpus=1:mem=40GB",  # Specify resources
    account="uwis0040",  # Input your project ID here / THIS WILL BE DIFFERENT FOR YOU 
    walltime="02:00:00",  # Amount of wall time
    interface="ext",  # Interface to use
)

# this is where we say that we want several of these compute systems,
# because we will have to deal with lots of data and can't just rely on one
cluster.adapt(maximum_jobs=24, minimum_jobs=8) # If you want to force everything to be quicker, 
# increase the number of minimum jobs, but sometimes then it will take a while until you get them assigned 
# (they have to queue), so it's a trade-off
client = distributed.Client(cluster)

# show the client that you have been assigned, you can click on the link and it will show you 
# a dashboard with all the tasks that have to be performed to do your calculation
client

/glade/work/jtcohen/envs/lib/python3.10/site-packages/distributed/node.py:179: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37923 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/37923/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/37923/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.196:33037,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/37923/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Load files

In [4]:
rad_val = 'andom'
MODE_output = f'/glade/derecho/scratch/jtcohen/MODE_files/output_r{rad_val}'
suffix = '_obj.txt'

files = sorted(glob.glob(f'{MODE_output}/i??????01/mode_m??E*{suffix}'))
len(files)

57600

In [5]:
n_ens = 20
years = list(range(1989, 2019))
months = ['02', '05', '08', '11']
ens = [f'{m+1:02d}' for m in range(n_ens)]
leads = list(range(24))
inits = [f'{year}{month}01' for year in years for month in months]
valids = [[init, (pd.to_datetime(init)+pd.DateOffset(months=lead)).strftime('%Y%m%d'), lead] for init in inits for lead in leads]
fnames = [f'{MODE_output}/i{init}/mode_m{en}E_{lead:06d}L_{valid}_000000V_000000A_obj.txt' for init, valid, lead in valids for en in ens]

In [6]:
all = set(fnames)
all.update(files)
len(all)

57600

In [7]:
print('Missing files:')
print(np.setdiff1d(fnames, files))
print()
print('Extra files:')
print(np.setdiff1d(files, fnames))

Missing files:
[]

Extra files:
[]


In [9]:
df_dask = dd.read_csv(files, sep=" ", skipinitialspace=True, assume_missing=True)

In [10]:
%%time
df_pandas = df_dask.compute()

CPU times: user 2min 8s, sys: 7.42 s, total: 2min 15s
Wall time: 3min 9s


In [12]:
df = df_pandas.reset_index()

# Save data

In [14]:
df['MEMBER'] = df['FCST_LEV'].str.lstrip('m').astype(int)
df['VALID'] = pd.to_datetime(df['FCST_VALID'], format='%Y%m%d_000000')
# df['INIT'] = [v - pd.tseries.offsets.DateOffset(months=m) for v, m in tqdm(zip(df['VALID'], df['FCST_LEAD']))]
df['INIT'] = pd.to_datetime(pd.DataFrame({
    'year': df['VALID'].dt.year+(df['VALID'].dt.month-df['FCST_LEAD']-1)//12,
    'month': (df['VALID'].dt.month-df['FCST_LEAD']-1)%12+1,
    'day': np.repeat(1, len(df['VALID']))}))
df['VALID_MONTH'] = df['VALID'].dt.month
df['INIT_MONTH'] = df['INIT'].dt.month
df['LEAD'] = df['FCST_LEAD']

df = df.drop(
    columns=[
        'VERSION',
        'MODEL',
        'GRID_RES',
        'DESC',
        'FCST_LEAD',
        'FCST_VALID',
        'FCST_ACCUM',
        'OBS_LEAD',
        'OBS_VALID',
        'OBS_ACCUM',
        'FCST_RAD',
        'FCST_THR',
        'OBS_RAD',
        'OBS_THR',
        'FCST_VAR',
        'FCST_UNITS',
        'FCST_LEV',
        'OBS_VAR',
        'OBS_UNITS',
        'OBS_LEV',
        'OBTYPE'],
)

In [15]:
MODE_save = '/glade/work/jtcohen/MODE_files_final/saved_output'
fout = f'{MODE_save}/mode_1989-2018_SMYLE_OISST_24L_ocetrac_r{rad_val}.pkl'
df.to_pickle(fout)
print('Saved to ', fout)

Saved to  /glade/work/jtcohen/MODE_files_final/saved_output/mode_1989-2018_SMYLE_OISST_24L_ocetrac_random.pkl
